# TideGBM quant walkthrough

Panel regression with **purged walk-forward**, **leak lint**, and **`objective="pairwise"`** for cross-sectional IC.

You bring features and labels — TideGBM handles time order and entity structure (TORB).

```bash
pip install tidegbm pandas numpy matplotlib
```

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tidegbm import TideRegressor, WalkForwardSplit, lint

rng = np.random.default_rng(42)
n_time, n_ent = 300, 16
rows = []
for t in range(n_time):
    market_factor = 0.15 * rng.normal()
    for e in range(n_ent):
        mom = rng.normal()
        vol = abs(rng.normal()) + 0.1
        rev = -0.3 * mom + rng.normal(scale=0.5)
        y = 0.35 * mom - 0.15 * vol + 0.2 * rev + market_factor + 0.2 * rng.normal()
        rows.append(
            {
                "ts": pd.Timestamp("2020-01-01") + pd.Timedelta(days=t),
                "symbol": f"SYM{e:02d}",
                "momentum": mom,
                "volatility": vol,
                "reversal": rev,
                "y": y,
            }
        )

df = pd.DataFrame(rows)
features = ["momentum", "volatility", "reversal"]
df.head()

## 1. Leak lint

Run **before** training. Strict mode (default at `fit`) blocks lookahead features.

In [ ]:
report = lint(df[features], df["y"], df["ts"])
print(report)
assert report.ok(), "Fix leakage before continuing"

## 2. Purged walk-forward

- **`embargo=1`**: one-day gap after train before test
- **`horizon=1`**: label is next-day return (in timestamp units)
- **`objective="pairwise"`**: optimize within-date ranking → cross-sectional IC

In [ ]:
wf = WalkForwardSplit(n_splits=5, embargo=1, purge=1, min_train_fraction=0.25)
fold_ics = []
models = []

for fold, (train_idx, test_idx) in enumerate(wf.split(df, timestamp=df["ts"])):
    tr, te = df.iloc[train_idx], df.iloc[test_idx]
    model = TideRegressor(
        n_estimators=80,
        learning_rate=0.08,
        max_depth=6,
        horizon=1,
        objective="pairwise",
        random_state=fold,
        verbose=0,
    )
    model.fit(
        tr[features],
        tr["y"],
        timestamp=tr["ts"],
        entity=tr["symbol"],
        feature_groups={"risk": ["volatility"], "alpha": ["momentum", "reversal"]},
    )
    pred = model.predict(te[features])
    ic = float(np.corrcoef(pred, te["y"])[0, 1])
    fold_ics.append(ic)
    models.append(model)
    print(f"fold {fold}: IC={ic:.3f}  train={len(tr)}  test={len(te)}")

print(f"\nmean IC: {np.nanmean(fold_ics):.3f}")

## 3. IC by fold

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3))
ax.bar(range(len(fold_ics)), fold_ics, color="steelblue")
ax.axhline(np.nanmean(fold_ics), color="crimson", ls="--", label=f"mean={np.nanmean(fold_ics):.3f}")
ax.set_xlabel("fold")
ax.set_ylabel("IC (Pearson)")
ax.set_title("Walk-forward IC — TideGBM pairwise")
ax.legend()
plt.tight_layout()
plt.show()

## 4. Feature importance & model I/O

Use the last fold model. For **level fit** (R²), switch to `objective="mse"`.

In [ ]:
last = models[-1]
imp = last.feature_importances(kind="gain")
for name, score in sorted(zip(features, imp), key=lambda x: -x[1]):
    print(f"{name:12s} {score:.4f}")

last.save_model("/tmp/tidegbm_demo.tgbm")
loaded = TideRegressor.load_model("/tmp/tidegbm_demo.tgbm")
np.allclose(last.predict(df[features].tail(50)), loaded.predict(df[features].tail(50)))

## Next steps

- Public benchmark: `python benchmarks/markets.py --horizons 1,3,5,10`
- Full API: [`docs/API.md`](../docs/API.md)
- MSE vs pairwise: use `objective="mse"` when you care about absolute levels, not ranks